**Bài 1**: Tính điểm trung bình và tổng số lượt đánh giá cho mỗi phim
* **Mục tiêu**:
  * Tính điểm trung bình của mỗi phim.
  * Đếm tổng số lượt đánh giá.
  * Tìm phim có điểm trung bình cao nhất (chỉ xét những phim có ít nhất 50 lượt đánh giá).
* **Giải pháp**:
  * Bước 1: Đọc file movies.txt và tạo một map (MovieID → Title).
  * Bước 2: Đọc file ratings_1.txt và ratings_2.txt, map MovieID → (Rating, 1).
  * Bước 3: Reduce để tính tổng điểm và số lượt đánh giá.
  * Bước 4: Tính điểm trung bình, lọc ra phim có ít nhất 5 lượt đánh giá.
  * Bước 5: Tìm phim có điểm trung bình cao nhất.

In [6]:
import pandas as pd

# Step 1: Read movies.txt and create a map (MovieID → Title)
movies_df = pd.read_csv('/content/movies.txt', sep=',', header=None, engine='python', names=['MovieID', 'Title', 'Genres'])
movie_id_to_title = dict(zip(movies_df['MovieID'], movies_df['Title']))

print(f"Number of movies loaded: {len(movie_id_to_title)}")
# Display the first few mappings to verify
print("First 5 MovieID to Title mappings:")
for movie_id, title in list(movie_id_to_title.items())[:5]:
    print(f"  {movie_id}: {title}")

Number of movies loaded: 50
First 5 MovieID to Title mappings:
  1001: The Godfather (1972)
  1002: The Shawshank Redemption (1994)
  1003: Schindler's List (1993)
  1004: Raging Bull (1980)
  1005: Casablanca (1942)


In [7]:
# Step 2: Read ratings_1.txt and ratings_2.txt
ratings_df1 = pd.read_csv('/content/ratings_1.txt', sep=',', header=None, engine='python', names=['UserID', 'MovieID', 'Rating', 'Timestamp'])
ratings_df2 = pd.read_csv('/content/ratings_2.txt', sep=',', header=None, engine='python', names=['UserID', 'MovieID', 'Rating', 'Timestamp'])

# Combine the two ratings dataframes
all_ratings_df = pd.concat([ratings_df1, ratings_df2], ignore_index=True)

print(f"Total ratings loaded: {len(all_ratings_df)}")
print("First 5 rows of combined ratings:")
display(all_ratings_df.head())

Total ratings loaded: 184
First 5 rows of combined ratings:


,UserID,MovieID,Rating,Timestamp
0,7,1020,4.5,1577836800
1,23,1015,3.5,1577923200
2,45,1030,4.0,1578009600
3,12,1047,3.0,1578096000
4,38,1012,4.5,1578182400


In [8]:
# Step 3: Reduce to calculate sum of ratings and count of reviews for each movie
# Group by MovieID and sum the ratings, and count the occurrences (number of reviews)
movie_stats = all_ratings_df.groupby('MovieID').agg(
    total_rating=('Rating', 'sum'),
    review_count=('Rating', 'count')
).reset_index()

print("First 5 rows of movie statistics:")
display(movie_stats.head())

First 5 rows of movie statistics:


,MovieID,total_rating,review_count
0,1010,62.0,18
1,1012,8.0,2
2,1013,68.0,17
3,1015,30.5,7
4,1020,66.0,18


In [11]:
# Step 4: Calculate average rating and filter for movies with at least 5 reviews
movie_stats['average_rating'] = movie_stats['total_rating'] / movie_stats['review_count']

# Filter for movies with at least 5 reviews
filtered_movies_stats = movie_stats[movie_stats['review_count'] >= 5]

print(f"Number of movies with at least 5 reviews: {len(filtered_movies_stats)}")
print("First 5 rows of filtered movie statistics:")
display(filtered_movies_stats.head())

Number of movies with at least 5 reviews: 13
First 5 rows of filtered movie statistics:


,MovieID,total_rating,review_count,average_rating
0,1010,62.0,18,3.444444
2,1013,68.0,17,4.000000
3,1015,30.5,7,4.357143
4,1020,66.0,18,3.666667
5,1025,73.0,18,4.055556


In [12]:
# Step 5: Find the movie with the highest average score
highest_rated_movie = filtered_movies_stats.loc[filtered_movies_stats['average_rating'].idxmax()]

# Get the title of the movie
winning_movie_id = highest_rated_movie['MovieID']
winning_movie_title = movie_id_to_title.get(winning_movie_id, 'Unknown Title')

print(f"The movie with the highest average rating (with at least 5 reviews) is:")
print(f"  MovieID: {winning_movie_id}")
print(f"  Title: {winning_movie_title}")
print(f"  Average Rating: {highest_rated_movie['average_rating']:.2f}")
print(f"  Number of Reviews: {highest_rated_movie['review_count']}")

The movie with the highest average rating (with at least 5 reviews) is:
  MovieID: 1015.0
  Title: Sunset Boulevard (1950)
  Average Rating: 4.36
  Number of Reviews: 7.0


Bài 2: Phân tích đánh giá theo thể loại
* Mục tiêu:
  * Tính điểm trung bình của từng thể loại phim.
* Giải pháp
  * Bước 1: Tạo map (MovieID → List of Genres).
  * Bước 2: Map từ MovieID → Rating → (Genre, Rating).
  * Bước 3: Tính trung bình điểm đánh giá cho từng thể loại.

In [13]:
# Bước 1: Tạo map (MovieID → List of Genres)
# The 'Genres' column is a string with genres separated by '|', convert it to a list
movies_df['Genres_List'] = movies_df['Genres'].apply(lambda x: x.split('|'))

# Create a dictionary mapping MovieID to its list of genres
movie_id_to_genres = dict(zip(movies_df['MovieID'], movies_df['Genres_List']))

print("First 5 MovieID to Genres mappings:")
for movie_id, genres in list(movie_id_to_genres.items())[:5]:
    print(f"  {movie_id}: {genres}")

First 5 MovieID to Genres mappings:
  1001: ['Crime', 'Drama']
  1002: ['Drama']
  1003: ['Biography', 'Drama', 'History']
  1004: ['Biography', 'Drama', 'Sport']
  1005: ['Drama', 'Romance', 'War']


In [14]:
# Bước 2: Map từ MovieID → Rating → (Genre, Rating)
# Create a list to store (genre, rating) pairs
genre_ratings_list = []

# Iterate through each rating and expand it by genre
for index, row in all_ratings_df.iterrows():
    movie_id = row['MovieID']
    rating = row['Rating']
    genres = movie_id_to_genres.get(movie_id, [])

    for genre in genres:
        genre_ratings_list.append({'Genre': genre, 'Rating': rating})

# Convert the list to a DataFrame
genre_ratings_df = pd.DataFrame(genre_ratings_list)

print(f"Total genre-ratings combinations: {len(genre_ratings_df)}")
print("First 5 rows of genre-ratings combinations:")
display(genre_ratings_df.head())

Total genre-ratings combinations: 471
First 5 rows of genre-ratings combinations:


,Genre,Rating
0,Family,4.5
1,Sci-Fi,4.5
2,Drama,3.5
3,Film-Noir,3.5
4,Crime,4.0


In [15]:
# Bước 3: Tính trung bình điểm đánh giá cho từng thể loại
average_rating_by_genre = genre_ratings_df.groupby('Genre')['Rating'].mean().reset_index()

# Sort by average rating in descending order for better insights
average_rating_by_genre = average_rating_by_genre.sort_values(by='Rating', ascending=False)

print("Average rating for each genre:")
display(average_rating_by_genre)

Average rating for each genre:


,Genre,Rating
7,Film-Noir,4.357143
8,Horror,4.000000
9,Mystery,4.000000
6,Fantasy,3.862069
3,Crime,3.809524
4,Drama,3.757812
10,Sci-Fi,3.731481
0,Action,3.712963
11,Thriller,3.703704
5,Family,3.666667


Bài 3: Phân tích đánh giá theo giới tính
* Mục tiêu:
  * Tính điểm trung bình của mỗi phim theo giới tính.
* Giải pháp
  * Bước 1: Tạo map (UserID → Gender).
  * Bước 2: Join với ratings để thêm thông tin giới tính.
  * Bước 3: Tính trung bình rating cho mỗi phim theo từng giới tính.

In [16]:
# Bước 1: Tạo map (UserID → Gender)
users_df = pd.read_csv('/content/users.txt', sep=',', header=None, engine='python', names=['UserID', 'Gender', 'Age', 'Occupation', 'Zip-code'])
user_id_to_gender = dict(zip(users_df['UserID'], users_df['Gender']))

print("First 5 UserID to Gender mappings:")
for user_id, gender in list(user_id_to_gender.items())[:5]:
    print(f"  {user_id}: {gender}")

First 5 UserID to Gender mappings:
  1: M
  2: F
  3: M
  4: F
  5: M


In [17]:
# Bước 2: Join với ratings để thêm thông tin giới tính
# Add Gender information to the all_ratings_df
ratings_with_gender_df = all_ratings_df.copy()
ratings_with_gender_df['Gender'] = ratings_with_gender_df['UserID'].map(user_id_to_gender)

print(f"Total ratings with gender information: {len(ratings_with_gender_df)}")
print("First 5 rows of ratings with gender:")
display(ratings_with_gender_df.head())

Total ratings with gender information: 184
First 5 rows of ratings with gender:


,UserID,MovieID,Rating,Timestamp,Gender
0,7,1020,4.5,1577836800,M
1,23,1015,3.5,1577923200,M
2,45,1030,4.0,1578009600,M
3,12,1047,3.0,1578096000,F
4,38,1012,4.5,1578182400,F


Error: Runtime no longer has a reference to this dataframe, please re-run this cell and try again.


In [18]:
# Bước 3: Tính trung bình rating cho mỗi phim theo từng giới tính
average_rating_by_movie_gender = ratings_with_gender_df.groupby(['MovieID', 'Gender'])['Rating'].mean().unstack()

print("Average rating for each movie by gender:")
display(average_rating_by_movie_gender.head())

Average rating for each movie by gender:


Gender,F,M
MovieID,,
1010,3.3125,3.550000
1012,4.0000,NaN
1013,3.9375,4.055556
1015,4.5000,4.333333
1020,3.5500,3.812500


Bài 4: Phân tích đánh giá theo nhóm tuổi
* Mục tiêu:
  * Phân loại người dùng theo nhóm tuổi và tính điểm trung bình của mỗi phim theo từng nhóm.
* Giải pháp
  * Bước 1: Tạo map (UserID → Age Group).
  * Bước 2: Join với ratings để thêm nhóm tuổi.
  * Bước 3: Tính trung bình điểm đánh giá theo nhóm tuổi.

In [19]:
# Bước 1: Tạo map (UserID → Age Group)
# Define age bins and labels
age_bins = [0, 12, 17, 24, 34, 44, 54, 64, 100]
age_labels = ['0-12', '13-17', '18-24', '25-34', '35-44', '45-54', '55-64', '65+']

# Create Age Group column in users_df
users_df['Age_Group'] = pd.cut(users_df['Age'], bins=age_bins, labels=age_labels, right=False)

# Create a dictionary mapping UserID to its Age Group
user_id_to_age_group = dict(zip(users_df['UserID'], users_df['Age_Group']))

print("First 5 UserID to Age Group mappings:")
for user_id, age_group in list(user_id_to_age_group.items())[:5]:
    print(f"  {user_id}: {age_group}")

First 5 UserID to Age Group mappings:
  1: 25-34
  2: 35-44
  3: 35-44
  4: 18-24
  5: 25-34


In [20]:
# Bước 2: Join với ratings để thêm nhóm tuổi
# Add Age Group information to the all_ratings_df
ratings_with_age_group_df = all_ratings_df.copy()
ratings_with_age_group_df['Age_Group'] = ratings_with_age_group_df['UserID'].map(user_id_to_age_group)

print(f"Total ratings with age group information: {len(ratings_with_age_group_df)}")
print("First 5 rows of ratings with age group:")
display(ratings_with_age_group_df.head())

Total ratings with age group information: 184
First 5 rows of ratings with age group:


,UserID,MovieID,Rating,Timestamp,Age_Group
0,7,1020,4.5,1577836800,35-44
1,23,1015,3.5,1577923200,25-34
2,45,1030,4.0,1578009600,45-54
3,12,1047,3.0,1578096000,35-44
4,38,1012,4.5,1578182400,25-34


In [21]:
# Bước 3: Tính trung bình điểm đánh giá theo nhóm tuổi
average_rating_by_movie_age_group = ratings_with_age_group_df.groupby(['MovieID', 'Age_Group'])['Rating'].mean().unstack()

print("Average rating for each movie by age group:")
display(average_rating_by_movie_age_group.head())

Average rating for each movie by age group:


Age_Group,18-24,25-34,35-44,45-54,55-64
MovieID,,,,,
1010,NaN,3.600000,3.333333,3.25,4.5
1012,NaN,4.500000,3.500000,NaN,NaN
1013,4.5,3.785714,4.000000,4.50,NaN
1015,NaN,4.166667,4.500000,4.50,NaN
1020,4.0,3.500000,3.928571,3.50,3.0


Bài 5: Phân Tích Đánh Giá Theo Occupation (Nghề nghiệp) Của Người Dùng
* Mục tiêu:
  * Tính trung bình rating và tổng số lượt đánh giá cho từng Occupation.
* Giải pháp:
  * Tạo dictionary từ users.txt với mapping UserID → Occupation.
  * Với mỗi rating, gán thông tin Occupation theo UserID.
  * Phát hành cặp key-value với key là Occupation và value là (rating, 1).
  * Reduce để tính tổng điểm và số lượt cho mỗi Occupation, sau đó tính trung bình rating.

### Bước 1: Tạo dictionary từ users.txt với mapping UserID → Occupation

In [22]:
users_df = pd.read_csv('/content/users.txt', sep=',', header=None, engine='python', names=['UserID', 'Gender', 'Age', 'Occupation', 'Zip-code'])
user_id_to_occupation = dict(zip(users_df['UserID'], users_df['Occupation']))

print("First 5 UserID to Occupation mappings:")
for user_id, occupation in list(user_id_to_occupation.items())[:5]:
    print(f"  {user_id}: {occupation}")

First 5 UserID to Occupation mappings:
  1: 3
  2: 7
  3: 2
  4: 10
  5: 1


### Bước 2: Join với ratings để thêm thông tin Occupation

In [23]:
ratings_with_occupation_df = all_ratings_df.copy()
ratings_with_occupation_df['Occupation'] = ratings_with_occupation_df['UserID'].map(user_id_to_occupation)

print(f"Total ratings with occupation information: {len(ratings_with_occupation_df)}")
print("First 5 rows of ratings with occupation:")
display(ratings_with_occupation_df.head())

Total ratings with occupation information: 184
First 5 rows of ratings with occupation:


,UserID,MovieID,Rating,Timestamp,Occupation
0,7,1020,4.5,1577836800,12
1,23,1015,3.5,1577923200,14
2,45,1030,4.0,1578009600,12
3,12,1047,3.0,1578096000,8
4,38,1012,4.5,1578182400,11


### Bước 3: Tính trung bình rating và tổng số lượt đánh giá cho từng Occupation

In [24]:
occupation_stats = ratings_with_occupation_df.groupby('Occupation').agg(
    average_rating=('Rating', 'mean'),
    review_count=('Rating', 'count')
).reset_index()

# Sort by average rating in descending order for better insights
occupation_stats = occupation_stats.sort_values(by='average_rating', ascending=False)

print("Average rating and review count for each occupation:")
display(occupation_stats)

Average rating and review count for each occupation:


,Occupation,average_rating,review_count
0,1,4.250000,10
13,15,4.000000,8
11,12,4.000000,13
7,8,3.863636,11
12,14,3.857143,14
10,11,3.852941,17
5,6,3.727273,11
3,4,3.700000,5
1,2,3.690476,21
4,5,3.647059,17


Bài 6: Phân Tích Đánh Giá Theo Thời Gian
* Mục tiêu:
  * Tính tổng số lượt đánh giá và điểm trung bình cho mỗi năm.
* Giải pháp:
  * Đọc dữ liệu ratings (từ cả ratings_1.txt và ratings_2.txt).
  * Sử dụng hàm trợ giúp để chuyển đổi Timestamp (dạng Unix) thành năm (Year).
  * Với mỗi dòng ratings, phát hành cặp key-value với key là năm và value là (rating, 1).
  * Reduce để tính tổng điểm và số lượt cho mỗi năm, sau đó tính trung bình.

### Bước 1: Chuyển đổi Timestamp thành năm

In [25]:
# Convert Unix timestamp to datetime objects and extract the year
ratings_by_time_df = all_ratings_df.copy()
ratings_by_time_df['Year'] = pd.to_datetime(ratings_by_time_df['Timestamp'], unit='s').dt.year

print("First 5 rows of ratings with Year:")
display(ratings_by_time_df.head())

First 5 rows of ratings with Year:


,UserID,MovieID,Rating,Timestamp,Year
0,7,1020,4.5,1577836800,2020
1,23,1015,3.5,1577923200,2020
2,45,1030,4.0,1578009600,2020
3,12,1047,3.0,1578096000,2020
4,38,1012,4.5,1578182400,2020


### Bước 2: Tính tổng số lượt đánh giá và điểm trung bình cho mỗi năm

In [26]:
yearly_stats = ratings_by_time_df.groupby('Year').agg(
    total_reviews=('Rating', 'count'),
    average_rating=('Rating', 'mean')
).reset_index()

# Sort by year for chronological order
yearly_stats = yearly_stats.sort_values(by='Year', ascending=True)

print("Total reviews and average rating per year:")
display(yearly_stats)

Total reviews and average rating per year:


,Year,total_reviews,average_rating
0,2020,184,3.752717
